# ÉVALUATION DES SYSTÈMES DE RECOMMANDATION
## Content-Based vs Collaborative Filtering Item-Based

Ce notebook évalue les performances de 2 systèmes de recommandation:
1. **Content-Based**: Utilise les embeddings des articles
2. **Collaborative Filtering Item-Based**: Articles similaires

## IMPORTS ET CHARGEMENT DES DONNÉES

In [19]:
import pandas as pd
import numpy as np
import os
import pickle
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix
from sklearn.metrics import precision_score, recall_score, ndcg_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import faiss
from tqdm import tqdm 
warnings.filterwarnings('ignore')


print("Imports réussis!")

Imports réussis!


## CHARGEMENT DES DONNÉES

In [13]:
# Chargement des fichiers
print("Chargement des données...\\n")

# Clics
all_files = os.listdir('data/clicks/')
df_all = pd.concat((pd.read_csv(f'data/clicks/{file}') for file in all_files), ignore_index=True)
df_all = df_all.sort_values('click_timestamp').reset_index(drop=True)

# Articles
df_articles = pd.read_csv('data/articles_metadata.csv')

# Embeddings (pour content-based)
with open('data/articles_embeddings.pickle', 'rb') as f:
    embeddings = pickle.load(f)
    
article_ids = df_articles['article_id'].values

idx_to_article = {i: aid for i, aid in enumerate(article_ids)}
article_to_idx = {aid: i for i, aid in enumerate(article_ids)}

print("Mapping OK:")
print(len(article_to_idx), "articles")

print(f" Données chargées:")
print(f"   • Clics totaux: {len(df_all):,}")
print(f"   • Utilisateurs uniques: {df_all['user_id'].nunique():,}")
print(f"   • Articles uniques: {df_all['click_article_id'].nunique():,}")
print(f"   • Articles metadata: {len(df_articles):,}")
print(f"   • Embeddings shape: {embeddings.shape}")
print(f"\n Plage temporelle:")
print(f"   • Min timestamp: {df_all['click_timestamp'].min()}")
print(f"   • Max timestamp: {df_all['click_timestamp'].max()}")

Chargement des données...\n
Mapping OK:
364047 articles
 Données chargées:
   • Clics totaux: 2,988,181
   • Utilisateurs uniques: 322,897
   • Articles uniques: 46,033
   • Articles metadata: 364,047
   • Embeddings shape: (364047, 250)

 Plage temporelle:
   • Min timestamp: 1506826800026
   • Max timestamp: 1510603454886


## TRAIN/TEST SPLIT (Temporal Split)

In [3]:
df_all

,user_id,session_id,session_start,session_size,click_article_id,click_timestamp,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type
0,59,1506826329267796,1506826329000,2,234853,1506826800026,4,3,2,1,21,1
1,79,1506826486139816,1506826486000,2,159359,1506826801702,4,3,2,1,13,1
2,154,1506826793323891,1506826793000,2,96663,1506826804207,4,3,2,1,25,7
3,111,1506826621773848,1506826621000,2,202436,1506826814140,4,3,2,1,9,7
4,70,1506826416120807,1506826416000,3,119592,1506826818863,4,3,2,1,13,2
...,...,...,...,...,...,...,...,...,...,...,...,...
2988176,201738,1507719721359504,1507719721000,42,224148,1509798422502,4,1,17,1,13,2
2988177,252642,1507932701183006,1507932701000,10,207672,1510093882860,4,1,17,1,20,2
2988178,252642,1507932701183006,1507932701000,10,96333,1510093912860,4,1,17,1,20,2
2988179,320431,1508183187238264,1508183187000,8,203538,1510603424886,4,1,17,1,2,2


In [4]:
df_articles

,article_id,category_id,created_at_ts,publisher_id,words_count
0,0,0,1513144419000,0,168
1,1,1,1405341936000,0,189
2,2,1,1408667706000,0,250
3,3,1,1408468313000,0,230
4,4,1,1407071171000,0,162
...,...,...,...,...,...
364042,364042,460,1434034118000,0,144
364043,364043,460,1434148472000,0,463
364044,364044,460,1457974279000,0,177
364045,364045,460,1515964737000,0,126


In [5]:
print(embeddings.shape)
embeddings

(364047, 250)


array([[-0.16118301, -0.95723313, -0.13794445, ..., -0.231686  ,
         0.5974159 ,  0.40962312],
       [-0.52321565, -0.974058  ,  0.73860806, ...,  0.18282819,
         0.39708954, -0.83436364],
       [-0.61961854, -0.9729604 , -0.20736018, ..., -0.44758022,
         0.8059317 , -0.28528407],
       ...,
       [-0.25139043, -0.9762427 ,  0.58609664, ..., -0.14372464,
         0.06809307, -0.7050104 ],
       [ 0.22434181, -0.92328775, -0.38174152, ...,  0.6871319 ,
        -0.5315117 ,  0.01072566],
       [-0.25713393, -0.9946313 ,  0.9837918 , ...,  0.98387307,
        -0.8381829 , -0.1792827 ]], dtype=float32)

In [14]:
# Split temporel: 80% train, 20% test
# Important pour un système de recommandation
split_idx = int(len(df_all) * 0.8)

df_train = df_all.iloc[:split_idx].copy()
df_test = df_all.iloc[split_idx:].copy()

print(f"Train/Test Split (Temporal):")
print(f"   • Train: {len(df_train):,} clics ({len(df_train)/len(df_all)*100:.1f}%)")
print(f"   • Test: {len(df_test):,} clics ({len(df_test)/len(df_all)*100:.1f}%)")
print(f"\n Utilisateurs:")
print(f"   • Train: {df_train['user_id'].nunique():,} uniques")
print(f"   • Test: {df_test['user_id'].nunique():,} uniques")
print(f"   • Overlap: {len(set(df_train['user_id']).intersection(set(df_test['user_id']))):,} utilisateurs dans les 2")

Train/Test Split (Temporal):
   • Train: 2,390,544 clics (80.0%)
   • Test: 597,637 clics (20.0%)

 Utilisateurs:
   • Train: 291,111 uniques
   • Test: 130,843 uniques
   • Overlap: 99,057 utilisateurs dans les 2


## CONSTRUCTION: COLLABORATIVE FILTERING ITEM-BASED

In [15]:
print("Construction du système Collaborative Filtering Item-Based...\n")

import numpy as np
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
import faiss

# ============================================================================
# 1. MAPPINGS
# ============================================================================

user_unique_train = df_train['user_id'].unique()
article_unique_train = df_train['click_article_id'].unique()

user_to_idx = {uid: idx for idx, uid in enumerate(user_unique_train)}
article_to_idx = {aid: idx for idx, aid in enumerate(article_unique_train)}

idx_to_user = {idx: uid for uid, idx in user_to_idx.items()}
idx_to_article = {idx: aid for aid, idx in article_to_idx.items()}

n_users = len(user_unique_train)
n_items = len(article_unique_train)

print(f"Users: {n_users:,} | Items: {n_items:,}")

# ============================================================================
# 2. USER-ITEM MATRIX (SPARSE)
# ============================================================================

rows = [user_to_idx[uid] for uid in df_train['user_id']]
cols = [article_to_idx[aid] for aid in df_train['click_article_id']]
data = np.ones(len(df_train), dtype=np.float32)

user_item_matrix = csr_matrix(
    (data, (rows, cols)),
    shape=(n_users, n_items)
)

print(f"User-item matrix: {user_item_matrix.shape}")

# ============================================================================
# 3. EMBEDDINGS ITEMS (VERSION PROPRE)
# ============================================================================

print("\nConstruction embeddings items (SVD)...")

# réduction dimensionnelle (sinon trop lourd)
svd = TruncatedSVD(
    n_components=128,
    random_state=42
)

item_embeddings = svd.fit_transform(user_item_matrix.T)

print(f"Embedding shape after SVD: {item_embeddings.shape}")

# ============================================================================
# 4. FORMAT FAISS
# ============================================================================

item_embeddings = item_embeddings.astype(np.float32)

# obligatoire pour FAISS
item_embeddings = np.ascontiguousarray(item_embeddings)

# normalisation cosine
faiss.normalize_L2(item_embeddings)

# ============================================================================
# 5. INDEX FAISS
# ============================================================================

print("\nConstruction index FAISS...")

index_faiss_items = faiss.IndexFlatIP(item_embeddings.shape[1])
index_faiss_items.add(item_embeddings)

print(" FAISS index construit")
print(f"   • Items indexés: {index_faiss_items.ntotal:,}")
print(f"   • Dimension embedding: {item_embeddings.shape[1]}")

Construction du système Collaborative Filtering Item-Based...

Users: 291,111 | Items: 36,439
User-item matrix: (291111, 36439)

Construction embeddings items (SVD)...
Embedding shape after SVD: (36439, 128)

Construction index FAISS...
 FAISS index construit
   • Items indexés: 36,439
   • Dimension embedding: 128


## CONSTRUCTION: CONTENT-BASED

In [16]:
import faiss
import numpy as np

clicks_grouped = df_train.groupby("user_id")["click_article_id"].apply(list).to_dict()


embeddings_raw = embeddings.astype(np.float32).copy()


item_embeddings_cb = embeddings.astype(np.float32).copy()
faiss.normalize_L2(item_embeddings_cb)

index_faiss_cb = faiss.IndexFlatIP(item_embeddings_cb.shape[1])
index_faiss_cb.add(item_embeddings_cb)

# Mapping article_id → index dans embeddings (basé sur df_articles)
article_to_idx_cb = {aid: i for i, aid in enumerate(article_ids)}

def build_user_vector(user_id):
    if user_id not in clicks_grouped:
        return None

    articles = clicks_grouped[user_id]
    indices = [article_to_idx_cb[a] for a in articles if a in article_to_idx_cb]

    if not indices:
        return None

    #  embeddings BRUTS pour la moyenne 
    vecs = embeddings_raw[indices]
    mean_vec = np.mean(vecs, axis=0).astype(np.float32)

    #  Normaliser 
    norm = np.linalg.norm(mean_vec)
    if norm > 0:
        mean_vec = mean_vec / norm

    return mean_vec

def recommend_content_based(user_id, n_recommendations=5):
    k = n_recommendations

    user_vec = build_user_vector(user_id)
    if user_vec is None:
        return []

    user_vec = user_vec.reshape(1, -1)
    seen_article_ids = set(clicks_grouped.get(user_id, []))

    D, I = index_faiss_cb.search(user_vec, k + len(seen_article_ids) + 50)

    recs = []
    for faiss_idx in I[0]:
        if faiss_idx < 0 or faiss_idx >= len(article_ids):
            continue

        article_id = article_ids[faiss_idx]  # index FAISS → article_id réel

        if article_id in seen_article_ids:
            continue

        recs.append(article_id)

        if len(recs) == k:
            break

    return recs

test_by_user = df_test.groupby("user_id")["click_article_id"].apply(set).to_dict()
test_users = list(set(df_test.user_id).intersection(df_train.user_id))

print(" Content-Based construit")

 Content-Based construit


## FONCTIONS DE RECOMMANDATION

In [ ]:
def evaluate_one_user(user_id, recommendations_dict, test_by_user, k=5):
    """
    Évalue les recommandations pour un utilisateur.
    
    Returns:
        (precision, recall, hit, ndcg) ou None si pas évaluable
    """
    recommendations = recommendations_dict.get(user_id, [])
    true_items = test_by_user.get(user_id, set())

    #  Pas évaluable 
    if len(recommendations) == 0 or len(true_items) == 0:
        return None

    #  Tronquer à k au cas où 
    recommendations = recommendations[:k]
    rec_set = set(recommendations)

    #  Hits 
    hits_in_recs = len(rec_set & true_items)

    #  Precision@k
    precision = hits_in_recs / k

    #  Recall@k 
    recall = hits_in_recs / len(true_items)

    #  Hit Rate : au moins 1 bon article dans les k 
    hit = 1 if hits_in_recs > 0 else 0

    #  NDCG@k : pénalise les bons articles en bas de liste 
    dcg = 0.0
    for rank, article_id in enumerate(recommendations):
        if article_id in true_items:
            dcg += 1.0 / np.log2(rank + 2)  # +2 car rank commence à 0

    # IDCG : meilleur classement possible (tous les hits en tête)
    n_hits_possible = min(len(true_items), k)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(n_hits_possible))

    ndcg = dcg / idcg if idcg > 0 else 0.0

    return precision, recall, hit, ndcg

In [17]:

def recommend_item_based(user_id, n_recommendations=5, n_similar_items=100):

    if user_id not in user_to_idx:
        return []

    user_idx = user_to_idx[user_id]

    user_articles_idx = user_item_matrix[user_idx].nonzero()[1]

    if len(user_articles_idx) == 0:
        return []

    user_liked = set(idx_to_article[i] for i in user_articles_idx)

    scores = {}

    for item_idx in user_articles_idx:

        # QUERY VECTOR
        query = item_embeddings[item_idx].reshape(1, -1)

        # FAISS SEARCH
        distances, indices = index_faiss_items.search(
            query,
            n_similar_items + 1
        )

        for rank, sim_item_idx in enumerate(indices[0][1:]):

            article_id = idx_to_article[sim_item_idx]

            if article_id in user_liked:
                continue

            score = float(distances[0][rank + 1])

            scores[article_id] = scores.get(article_id, 0) + score

    if not scores:
        return []

    top_items = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )[:n_recommendations]

    return [i for i, _ in top_items]



print("Fonctions de recommandation")

Fonctions de recommandation


## RÉSULTATS

In [18]:

# ============================================================================
# ÉVALUATION OPTIMISÉE SUR UTILISATEURS AVEC SIGNAL FORT
# ============================================================================

import numpy as np
import pandas as pd
import time
from collections import defaultdict
from joblib import Parallel, delayed
from tqdm import tqdm

print("\n" + "="*80)
print(" Évaluation sur utilisateurs avec signal fort")
print("="*80 + "\n")
# ============================================================================
# FONCTION ÉVALUATION RAPIDE
# ============================================================================

def evaluate_fast(test_users, recommendations_dict, test_by_user, k=5):

    results = Parallel(n_jobs=N_JOBS, prefer="threads")(
        delayed(evaluate_one_user)(
            user_id, recommendations_dict, test_by_user, k
        )
        for user_id in test_users
    )

    results = [r for r in results if r is not None]

    if len(results) == 0:
        return {
            'precision_at_k': 0,
            'recall_at_k': 0,
            'hit_rate': 0,
            'ndcg_at_k': 0,
        }

    precisions, recalls, hits, ndcgs = zip(*results)

    return {
        'precision_at_k': np.mean(precisions),
        'recall_at_k':    np.mean(recalls),
        'hit_rate':       np.mean(hits),
        'ndcg_at_k':      np.mean(ndcgs),
        'n_users_eval':   len(results),
    }
# ============================================================================
# PARAMÈTRES
# ============================================================================

MIN_TEST_CLICKS = 3
TOP_K = 5
N_JOBS = -1   # utilise tous les CPU
n_users = 10000   # nombre d'utilisateurs à évaluer (pour test rapide)
# ============================================================================
# FILTRAGE UTILISATEURS AVEC SIGNAL SUFFISANT
# ============================================================================

clicks_per_user_test = df_test.groupby('user_id').size()

users_with_signal = clicks_per_user_test[
    clicks_per_user_test >= MIN_TEST_CLICKS
].index.to_numpy()


np.random.seed(42)
users_with_signal = np.random.choice(
    users_with_signal,
    min(n_users, len(users_with_signal)),
    replace=False
)

print(f" Utilisateurs sélectionnés: {len(users_with_signal):,}")

# ============================================================================
# SUBSET TEST
# ============================================================================

df_test_subset = df_test[
    df_test['user_id'].isin(users_with_signal)
]

print(f"\n STATISTIQUES DU SUBSET:")
print(f"   • Clics totaux: {len(df_test_subset):,}")
print(f"   • Clics moyens/user: {df_test_subset.groupby('user_id').size().mean():.2f}")
print(f"   • Articles uniques: {df_test_subset['click_article_id'].nunique():,}")

# ============================================================================
# CONSTRUCTION RAPIDE GROUND TRUTH
# ============================================================================

print("\n Construction du ground truth...")



test_by_user = (
    df_test_subset.groupby('user_id')['click_article_id']
    .apply(set)
    .to_dict()
)

test_users = users_with_signal

print(f" Ground truth construit pour {len(test_users):,} utilisateurs")

# ============================================================================
# PRÉ-CALCUL RECOMMANDATIONS (ÉNORME GAIN DE TEMPS)
# ============================================================================

def precompute_recommendations(users, recommendation_func, k=5):

    recommendations_dict = {}

    for user_id in tqdm(users, desc="Pré-calcul recommandations"):

        try:
            recs = recommendation_func(
                user_id,
                n_recommendations=k
            )

            recommendations_dict[user_id] = recs

        except Exception:
            recommendations_dict[user_id] = []

    return recommendations_dict

print("\n Pré-calcul Item-Based...")
start_ib = time.time()

precomputed_ib = precompute_recommendations(
    test_users,
    recommend_item_based,
    k=TOP_K
)

print(f"Item-Based pré-calculé en {(time.time()-start_ib):.2f}s")
# ============================================================================
# ÉVALUATION ITEM-BASED
# ============================================================================

print("\n Évaluation Item-Based optimisée...")

start_eval_ib = time.time()

metrics_ib_subset = evaluate_fast(
    test_users,
    precomputed_ib,
    test_by_user,
    k=TOP_K
)


print(f"Item-Based évalué en {(time.time()-start_eval_ib):.2f}s")
print(f"\n Item-Based CF (Signal) - RÉSULTATS")
print(f"   • Precision@{TOP_K}: {metrics_ib_subset['precision_at_k']:.4f}")
print(f"   • Recall@{TOP_K}:    {metrics_ib_subset['recall_at_k']:.4f}")
print(f"   • Hit Rate:         {metrics_ib_subset['hit_rate']:.4f}")

print("\n Pré-calcul Content-Based...")
start_cb = time.time()

precomputed_cb = precompute_recommendations(
    test_users,
    recommend_content_based,
    k=TOP_K
)

print(f"Content-Based pré-calculé en {(time.time()-start_cb):.2f}s")




print(f"Terminé en {(time.time()-start_eval_ib):.2f}s")

# ============================================================================
# ÉVALUATION CONTENT-BASED
# ============================================================================

print("\n Évaluation Content-Based optimisée...")

start_eval_cb = time.time()

metrics_cb_subset = evaluate_fast(
    test_users,
    precomputed_cb,
    test_by_user,
    k=TOP_K
)

print(f"Terminé en {(time.time()-start_eval_cb):.2f}s")

# ============================================================================
# AFFICHAGE RÉSULTATS
# ============================================================================

print(f"\n{'='*80}")
print("RÉSULTATS SUR UTILISATEURS AVEC SIGNAL FORT")
print(f"{'='*80}")

results_comparison = pd.DataFrame({

    'Métrique': [
        'Precision@5',
        'Recall@5',
        'Hit Rate'
    ],

    # 'Item-Based (FULL)': [
    #     f"{metrics_item_based['precision_at_k']:.4f}",
    #     f"{metrics_item_based['recall_at_k']:.4f}",
    #     f"{metrics_item_based['hit_rate']:.4f}",
    # ],

    'Item-Based (Signal)': [
        f"{metrics_ib_subset['precision_at_k']:.4f}",
        f"{metrics_ib_subset['recall_at_k']:.4f}",
        f"{metrics_ib_subset['hit_rate']:.4f}",
    ],

    # 'Content-Based (FULL)': [
    #     f"{metrics_content_based['precision_at_k']:.4f}",
    #     f"{metrics_content_based['recall_at_k']:.4f}",
    #     f"{metrics_content_based['hit_rate']:.4f}",
    # ],

    'Content-Based (Signal)': [
        f"{metrics_cb_subset['precision_at_k']:.4f}",
        f"{metrics_cb_subset['recall_at_k']:.4f}",
        f"{metrics_cb_subset['hit_rate']:.4f}",
    ]
})

print(results_comparison.to_string(index=False))

# ============================================================================
# ANALYSE
# ============================================================================


# improvement_ib = (
#     metrics_ib_subset['hit_rate']
#     - metrics_item_based['hit_rate']
# )

# improvement_cb = (
#     metrics_cb_subset['hit_rate']
#     # - metrics_content_based['hit_rate']
# )


print(f"\n - ITEM-BASED:")
# print(f"   Hit Rate FULL   : {metrics_item_based['hit_rate']:.4f}")
print(f"   Hit Rate SIGNAL : {metrics_ib_subset['hit_rate']:.4f}")
# print(f"   Gain            : {improvement_ib:+.4f}")

print(f"\n - CONTENT-BASED:")
# print(f"   Hit Rate FULL   : {metrics_content_based['hit_rate']:.4f}")
print(f"   Hit Rate SIGNAL : {metrics_cb_subset['hit_rate']:.4f}")
# print(f"   Gain            : {improvement_cb:+.4f}")




print("\n" + "="*80)
print(" ÉVALUATION TERMINÉE")
print("="*80)


 Évaluation sur utilisateurs avec signal fort

 Utilisateurs sélectionnés: 10,000

 STATISTIQUES DU SUBSET:
   • Clics totaux: 67,944
   • Clics moyens/user: 6.79
   • Articles uniques: 4,400

 Construction du ground truth...
 Ground truth construit pour 10,000 utilisateurs

 Pré-calcul Item-Based...


Pré-calcul recommandations: 100%|██████████| 10000/10000 [01:54<00:00, 87.64it/s]


Item-Based pré-calculé en 114.11s

 Évaluation Item-Based optimisée...
Item-Based évalué en 0.38s

 Item-Based CF (Signal) - RÉSULTATS
   • Precision@5: 0.0019
   • Recall@5:    0.0013
   • Hit Rate:         0.0084

 Pré-calcul Content-Based...


Pré-calcul recommandations: 100%|██████████| 10000/10000 [02:31<00:00, 66.17it/s]


Content-Based pré-calculé en 151.13s
Terminé en 151.50s

 Évaluation Content-Based optimisée...
Terminé en 0.34s

RÉSULTATS SUR UTILISATEURS AVEC SIGNAL FORT
   Métrique Item-Based (Signal) Content-Based (Signal)
Precision@5              0.0019                 0.0006
   Recall@5              0.0013                 0.0003
   Hit Rate              0.0084                 0.0031

 - ITEM-BASED:
   Hit Rate SIGNAL : 0.0084

 - CONTENT-BASED:
   Hit Rate SIGNAL : 0.0031

 ÉVALUATION TERMINÉE
